# 📈 Forecasting com Random Forest

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | Random Forest — Ensemble de Árvores de Decisão (regressão de séries temporais via lags) |
| **Biblioteca** | `scikit-learn` 1.7.2 — `RandomForestRegressor` |
| **Hiperparâmetros configurados** | `n_lags=12`, `SEED=42`; grid manual com 5 configurações: `n_estimators` ∈ {200, 300, 500}, `max_depth` ∈ {4, 5, 6, 8, None}, `min_samples_split` ∈ {3, 4, 5, 10}, `min_samples_leaf` ∈ {1, 2, 3, 4}, `max_features` ∈ {`'sqrt'`, 0.8} |
| **Busca de hiperparâmetros** | Sim — grid manual (5 configs), selecionada por MAPE em holdout temporal (~20% do treino) |
| **Critério de seleção** | MAPE (holdout temporal) |
| **Séries utilizadas** | 29 séries com treino ≥ 36 observações (`len(train_series) < 36`) |
| **Horizonte** | 3 meses (`prediction_length = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses |
| **Reprodutibilidade** | `SEED = 42` (`random.seed(42)` + `np.random.seed(42)` + `random_state=SEED`) |
| **Referência** | Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. |

---

In [9]:
# ── Semente global para reprodutibilidade ──
import random, numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"🔒 Seed fixada: {SEED}")

🔒 Seed fixada: 42


## 1. Importação de Bibliotecas

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas!")

✅ Bibliotecas carregadas!


## 2. Carregamento dos Dados

In [11]:
# Carregar base econômica
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)

# Lista de todas as séries (excluindo séries removidas do pipeline de coleta)
ALL_SERIES = [col for col in df.columns if col not in ['PIM', 'IPCA_mensal']]

print(f"📊 Base carregada: {len(df)} observações")
print(f"📈 Séries disponíveis: {len(ALL_SERIES)}")
print(f"📅 Período: {df.index.min().strftime('%Y-%m')} a {df.index.max().strftime('%Y-%m')}")
df.head()

📊 Base carregada: 108 observações
📈 Séries disponíveis: 29
📅 Período: 2017-01 a 2025-12


,IBC_Br,Selic,Cambio_USDBRL,Desemprego,Brent_USD,Soja_USD,Minerio_USD,Ibovespa,ICC_FGV,Credito_Total,...,INCC,ICE_Empresarial,Housing_Starts_EUA,Dollar_Index_Fed,PMS_Volume,M2,Divida_PIB,Vendas_Varejo,NUCI_FGV,SP500
2017-01-01,90.01742,13.17,3.1270,12.7,55.25,379.589979,80.818182,64671.0,102.25,1537976.0,...,0.41,90.8,1190.0,116.2241,86.70229,5.842420e+09,46.46,89.14,73.2,2278.870117
2017-02-01,90.35700,12.82,3.0993,13.3,53.36,380.872624,88.950000,66662.0,113.80,1535492.0,...,0.65,88.4,1271.0,116.0032,82.92088,5.861693e+09,47.26,82.01,73.7,2363.639893
2017-03-01,99.05191,12.15,3.1684,13.9,52.20,366.095056,87.195652,64984.0,109.38,1540450.0,...,0.16,99.6,1190.0,114.7938,89.89053,5.936526e+09,47.53,88.52,73.3,2362.719971
2017-04-01,93.20176,11.59,3.1984,13.7,49.46,347.861310,70.400000,65403.0,109.01,1530470.0,...,-0.02,92.4,1146.0,114.7877,85.94189,5.925396e+09,47.48,88.31,73.4,2384.199951
2017-05-01,94.63310,11.15,3.2437,13.4,49.40,350.179987,61.630435,62711.0,103.49,1526937.0,...,0.63,105.0,1157.0,113.2956,89.81432,5.947256e+09,48.01,90.43,74.0,2411.800049


In [12]:
# Parâmetros de previsão
prediction_length = 3
HORIZON = prediction_length   # Alias para compatibilidade  # 3 meses à frente
context_length = 12    # usar últimos 12 meses como contexto
n_lags = 12            # número de lags para features

print(f"⚙️ Configuração:")
print(f"   - Horizonte de previsão: {prediction_length} meses")
print(f"   - Contexto mínimo: {context_length} meses")
print(f"   - Número de lags: {n_lags}")

⚙️ Configuração:
   - Horizonte de previsão: 3 meses
   - Contexto mínimo: 12 meses
   - Número de lags: 12


## 3. Configuração do Experimento

In [ ]:
# ============================================================
# Funções auxiliares para Random Forest
# ============================================================

def create_features(series_values, n_lags=12):
    """
    Cria features univariadas para o Random Forest a partir de uma série temporal.

    Para cada ponto t, gera um vetor de features:
      - n_lags lags: [y(t-n_lags), ..., y(t-1)]
      - MA3: média móvel dos últimos 3 meses (tendência de curto prazo)
      - MA6: média móvel dos últimos 6 meses (tendência de médio prazo)
      - STD6: desvio padrão dos últimos 6 meses (proxy de volatilidade)
    """
    X, y = [], []
    for i in range(n_lags, len(series_values)):
        lags = list(series_values[i - n_lags:i])
        ma3 = np.mean(series_values[max(0, i - 3):i])
        ma6 = np.mean(series_values[max(0, i - 6):i])
        std6 = np.std(series_values[max(0, i - 6):i], ddof=1) if i >= 6 else 0.0
        X.append(lags + [ma3, ma6, std6])
        y.append(series_values[i])
    return np.array(X), np.array(y)


def pred_features(buffer, n_lags=12):
    """
    Constrói o vetor de features para previsão recursiva.
    Usa o buffer atualizado (incluindo previsões anteriores).
    """
    s = np.array(buffer)
    lags = list(s[-n_lags:])
    ma3 = float(np.mean(s[-3:]))
    ma6 = float(np.mean(s[-6:]))
    std6 = float(np.std(s[-6:], ddof=1)) if len(s) >= 6 else 0.0
    return np.array(lags + [ma3, ma6, std6])


def optimize_rf_params(train_series, prediction_length=3, n_lags=12):
    """
    Otimiza hiperparâmetros do Random Forest usando validação temporal interna.
    
    Separa os últimos 20% do treino como validação e testa diferentes combinações
    de hiperparâmetros, selecionando a que minimiza o MAPE na validação.
    """
    values = train_series.values
    X_all, y_all = create_features(values, n_lags)
    
    # Validação temporal: últimos 20% como validação
    val_n = max(5, int(len(X_all) * 0.2))
    X_tr, y_tr = X_all[:-val_n], y_all[:-val_n]
    X_val, y_val = X_all[-val_n:], y_all[-val_n:]
    
    # Normalização
    from sklearn.preprocessing import StandardScaler
    sc_X = StandardScaler()
    sc_y = StandardScaler()
    X_tr_s = sc_X.fit_transform(X_tr)
    y_tr_s = sc_y.fit_transform(y_tr.reshape(-1, 1)).flatten()
    X_val_s = sc_X.transform(X_val)
    y_val_s = sc_y.transform(y_val.reshape(-1, 1)).flatten()
    
    best_mape = float('inf')
    best_params = {'n_estimators': 200, 'max_depth': 5, 'random_state': 42}
    
    # Grid search simples mas eficaz
    for n_est in [100, 200, 300]:
        for depth in [3, 5, 7, None]:
            for min_samples in [2, 5]:
                params = {
                    'n_estimators': n_est,
                    'max_depth': depth,
                    'min_samples_split': min_samples,
                    'min_samples_leaf': max(1, min_samples // 2),
                    'random_state': 42,
                    'n_jobs': -1,
                }
                model = RandomForestRegressor(**params)
                model.fit(X_tr_s, y_tr_s)
                preds_s = model.predict(X_val_s)
                preds = sc_y.inverse_transform(preds_s.reshape(-1, 1)).flatten()
                y_val_orig = sc_y.inverse_transform(y_val_s.reshape(-1, 1)).flatten()
                mape = np.mean(np.abs((y_val_orig - preds) / (y_val_orig + 1e-8))) * 100
                
                if mape < best_mape:
                    best_mape = mape
                    best_params = params
    
    return best_params


def forecast_with_random_forest(train_series, test_series, prediction_length=3, n_lags=12, rf_params=None):
    """
    Walk-forward multi-step com Random Forest.
    
    A cada janela de prediction_length meses, retreina o modelo com janela expansível
    e faz previsão recursiva para os próximos meses.
    
    Returns:
        forecasts: lista de previsões
        actuals: lista de valores reais
        dates: lista de datas correspondentes
        mapes: lista de MAPE por janela
        maes: lista de MAE por janela
        rmses: lista de RMSE por janela
    """
    from sklearn.preprocessing import StandardScaler
    
    if rf_params is None:
        rf_params = {'n_estimators': 200, 'max_depth': 5, 'random_state': 42, 'n_jobs': -1}
    
    train_vals = train_series.values
    test_vals = test_series.values
    
    all_forecasts, all_actuals, all_dates = [], [], []
    all_mapes, all_maes, all_rmses = [], [], []
    
    for i in range(0, len(test_vals), prediction_length):
        n_steps = min(prediction_length, len(test_vals) - i)
        
        # Janela expansível
        if i > 0:
            cur_train = np.concatenate([train_vals, test_vals[:i]])
        else:
            cur_train = train_vals
        
        # Cria features e normaliza
        X_tr, y_tr = create_features(cur_train, n_lags)
        sc_X = StandardScaler()
        sc_y = StandardScaler()
        X_tr_s = sc_X.fit_transform(X_tr)
        y_tr_s = sc_y.fit_transform(y_tr.reshape(-1, 1)).flatten()
        
        # Treina modelo
        model = RandomForestRegressor(**rf_params)
        model.fit(X_tr_s, y_tr_s)
        
        # Previsão recursiva
        buf = list(cur_train)
        preds_window = []
        for h in range(n_steps):
            feat = pred_features(buf, n_lags)
            yp_s = model.predict(sc_X.transform(feat.reshape(1, -1)))
            yp = sc_y.inverse_transform(yp_s.reshape(-1, 1)).flatten()[0]
            preds_window.append(yp)
            buf.append(yp)
        
        actuals_window = test_vals[i:i + n_steps]
        dates_window = test_series.index[i:i + n_steps].tolist()
        
        # Métricas da janela
        pw = np.array(preds_window)
        aw = np.array(actuals_window)
        mae = np.mean(np.abs(aw - pw))
        rmse = np.sqrt(np.mean((aw - pw) ** 2))
        mape = np.mean(np.abs((aw - pw) / (aw + 1e-8))) * 100
        
        all_forecasts.extend(preds_window)
        all_actuals.extend(actuals_window)
        all_dates.extend(dates_window)
        all_mapes.append(mape)
        all_maes.append(mae)
        all_rmses.append(rmse)
    
    return all_forecasts, all_actuals, all_dates, all_mapes, all_maes, all_rmses


print("✅ Funções auxiliares definidas: create_features, pred_features, optimize_rf_params, forecast_with_random_forest")

In [ ]:
# Dicionário para armazenar resultados
all_results = {}
TEST_SIZE = 24  # Últimos 24 meses para teste (padrão de todos os modelos)

# Train/test split fixo (igual aos demais notebooks)
train_df = df.iloc[:-TEST_SIZE]
test_df = df.iloc[-TEST_SIZE:]

print("="*80)
print("🚀 INICIANDO PREVISÕES COM RANDOM FOREST (COM OTIMIZAÇÃO)")
print(f"   Train: {train_df.index[0].strftime('%Y-%m')} a {train_df.index[-1].strftime('%Y-%m')} ({len(train_df)} obs)")
print(f"   Test:  {test_df.index[0].strftime('%Y-%m')} a {test_df.index[-1].strftime('%Y-%m')} ({len(test_df)} obs)")
print("="*80)

for series_name in tqdm(ALL_SERIES, desc="Processando séries"):
    try:
        train_series = train_df[series_name].dropna()
        test_series = test_df[series_name].dropna()

        if len(train_series) < 36:
            print(f"⚠️ {series_name}: Poucos dados de treino ({len(train_series)} obs)")
            continue

        if len(test_series) == 0:
            print(f"⚠️ {series_name}: Sem dados de teste")
            continue

        # Fase 1: Otimizar hiperparâmetros usando APENAS dados de treino
        # (holdout temporal interno dos últimos ~20% do treino como validação)
        best_params = optimize_rf_params(
            train_series,
            prediction_length=prediction_length,
            n_lags=n_lags
        )

        # Fase 2: Avaliar no teste com os best_params selecionados
        forecasts, actuals, dates, mapes, maes, rmses = forecast_with_random_forest(
            train_series,
            test_series,
            prediction_length=prediction_length,
            n_lags=n_lags,
            rf_params=best_params
        )

        if mapes:
            all_results[series_name] = {
                'mape_mean': np.mean(mapes),
                'mae_mean': np.mean(maes),
                'rmse_mean': np.mean(rmses),
                'n_points': sum(len(a) if hasattr(a, '__len__') else 1 for a in [actuals][:1]) if not isinstance(actuals, list) else len(actuals),
                'best_params': best_params,
                'forecasts': forecasts,
                'actuals': actuals,
                'dates': dates,
                'mapes': mapes
            }
            print(f"✅ {series_name}: MAPE = {np.mean(mapes):.2f}% | Params: depth={best_params.get('max_depth')}, trees={best_params['n_estimators']}")
        else:
            print(f"❌ {series_name}: Sem resultados válidos")

    except Exception as e:
        print(f"❌ {series_name}: Erro - {str(e)[:80]}")

print("\n" + "="*80)
print(f"✅ CONCLUÍDO: {len(all_results)}/{len(ALL_SERIES)} séries processadas")
print("="*80)

🚀 INICIANDO PREVISÕES COM RANDOM FOREST (COM OTIMIZAÇÃO)
   Train: 2017-01 a 2023-12 (84 obs)
   Test:  2024-01 a 2025-12 (24 obs)


Processando séries:   3%|▎         | 1/29 [01:10<32:44, 70.16s/it]

✅ IBC_Br: MAPE = 2.34% | Params: depth=4, trees=300


Processando séries:   7%|▋         | 2/29 [02:01<26:37, 59.18s/it]

✅ Selic: MAPE = 7.67% | Params: depth=4, trees=300


...

✅ Ouro_USD: MAPE = 11.03% | Params: depth=8, trees=200


Processando séries:  59%|█████▊    | 17/29 [07:54<04:28, 22.40s/it]

✅ GasNatural_USD: MAPE = 20.61% | Params: depth=8, trees=200


Processando séries:  62%|██████▏   | 18/29 [08:16<04:03, 22.15s/it]

✅ Cobre_USD: MAPE = 8.75% | Params: depth=8, trees=200


## 4. Resultados e Métricas

In [ ]:
# Classificação por MAPE
def classificar(mape):
    if mape < 10:
        return '⭐ Excelente'
    elif mape < 20:
        return '✅ Muito Bom'
    elif mape < 30:
        return '👍 Bom'
    elif mape < 50:
        return '⚠️ Regular'
    else:
        return '❌ Difícil'

# Criar DataFrame com resultados no formato padrão (Serie, MAE, RMSE, MAPE, N_Pontos, Classificacao)
results_df = pd.DataFrame([
    {
        'Serie': name,
        'MAE': data['mae_mean'],
        'RMSE': data['rmse_mean'],
        'MAPE': data['mape_mean'],
        'N_Pontos': data['n_points'],
        'Classificacao': classificar(data['mape_mean'])
    }
    for name, data in all_results.items()
]).sort_values('MAPE')

# Exibir tabela
print("="*80)
print("📊 RANKING - RANDOM FOREST")
print("="*80)
print(f"\nModelo: Random Forest | Horizonte: {prediction_length} meses | Lags: {n_lags}\n")
print(results_df.to_string(index=False))

# Estatísticas gerais
print("\n" + "-"*80)
print("📈 ESTATÍSTICAS GERAIS:")
print(f"   Total de séries analisadas: {len(results_df)}")
print(f"   MAPE médio geral: {results_df['MAPE'].mean():.2f}%")
print(f"   Melhor série: {results_df.loc[results_df['MAPE'].idxmin(), 'Serie']} ({results_df['MAPE'].min():.2f}%)")
print(f"   Pior série: {results_df.loc[results_df['MAPE'].idxmax(), 'Serie']} ({results_df['MAPE'].max():.2f}%)")
print(f"   Séries com MAPE < 10%: {(results_df['MAPE'] < 10).sum()}")
print(f"   Séries com MAPE < 20%: {(results_df['MAPE'] < 20).sum()}")

📊 RANKING - RANDOM FOREST

Modelo: Random Forest | Horizonte: 3 meses | Lags: 12

             Serie          MAE         RMSE      MAPE  N_Pontos Classificacao
      Prod_Ind_USA 5.617699e-01 6.163085e-01  0.558977        24   ⭐ Excelente
           CPI_USA 2.968023e+00 3.055355e+00  0.932941        24   ⭐ Excelente
        Divida_PIB 1.065884e+00 1.183513e+00  1.700101        24   ⭐ Excelente
          NUCI_FGV 1.434805e+00 1.556211e+00  1.750928        24   ⭐ Excelente
  Dollar_Index_Fed 2.453712e+00 2.628872e+00  1.981090        24   ⭐ Excelente
    Massa_Salarial 7.199223e+03 7.596076e+03  2.075779        24   ⭐ Excelente
            IBC_Br 2.584135e+00 2.975569e+00  2.361870        24   ⭐ Excelente
        PMS_Volume 3.148737e+00 3.638682e+00  2.943159        24   ⭐ Excelente
     Credito_Total 1.117612e+05 1.194821e+05  2.985962        24   ⭐ Excelente
   ICE_Empresarial 3.438004e+00 3.887356e+00  3.226469        24   ⭐ Excelente
           ICC_FGV 4.229007e+00 4.897933e+00  3.4

## 5. Visualização: Ranking MAPE por Série

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = results_df.sort_values('MAPE')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df['Serie'])
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'Random Forest — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('randomforest_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: randomforest_mape_por_serie.png")

## 6. Visualização: Real vs. Projetado (Top 6 Séries)

In [ ]:
# ── Gráfico: Real vs. Projetado (Top 6 Séries por MAPE) ──
top6 = sorted_df.head(6)['Serie'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, sn in zip(axes.flatten(), top6):
    data = all_results[sn]

    # Valores reais (teste)
    ax.plot(data['dates'], data['actuals'], 'b-o',
            label='Real', markersize=4, linewidth=2)

    # Previsões do modelo
    ax.plot(data['dates'], data['forecasts'], 'r--s',
            label='Previsão', markersize=4, linewidth=2)

    ax.set_title(f"{sn}\nMAPE: {data['mape_mean']:.1f}%", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

axes.flatten()[0].legend(fontsize=8)
fig.suptitle('Random Forest — Real vs. Projetado (6 Melhores Séries)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('randomforest_previsoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: randomforest_previsoes.png")

## 7. Exportação de Resultados

In [ ]:
# Salvar resultados no formato padrão (Serie, MAE, RMSE, MAPE, N_Pontos, Classificacao)
results_df.to_csv('resultados_randomforest.csv', index=False)
print("💾 Resultados salvos em 'resultados_randomforest.csv'")

# Salvar previsões no formato padrão (Serie, Data, Previsao)
previsoes_rows = []
for series_name, data in all_results.items():
    last_forecasts = data['forecasts'][-prediction_length:]
    last_date = data['dates'][-1]
    future_dates = pd.date_range(start=last_date, periods=len(last_forecasts)+1, freq='MS')[1:]
    for dt, val in zip(future_dates, last_forecasts):
        previsoes_rows.append({'Serie': series_name, 'Data': dt.strftime('%Y-%m-%d'), 'Previsao': val})

df_prev = pd.DataFrame(previsoes_rows)
df_prev.to_csv('previsoes_randomforest.csv', index=False)
print("💾 Previsões salvas em 'previsoes_randomforest.csv'")

💾 Resultados salvos em 'resultados_randomforest.csv'
💾 Previsões salvas em 'previsoes_randomforest.csv'
